# 03 — Cost Analysis

Identifies cost drivers: which DRGs, diagnoses, procedures, and beneficiary profiles are associated with the highest spending.

Requires `cms_data.db` from `01_setup.ipynb`.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

In [ ]:
DB = Path("../cms_data.db")
conn = sqlite3.connect(DB)

---
## 1. Cost drivers by DRG (Inpatient)

In [ ]:
drg_cost = pd.read_sql("""
SELECT
  CLM_DRG_CD as drg,
  COUNT(*) as num_claims,
  ROUND(SUM(CLM_PMT_AMT), 0) as total_cost,
  ROUND(AVG(CLM_PMT_AMT), 0) as avg_cost_per_claim,
  ROUND(AVG(CLM_UTLZTN_DAY_CNT), 1) as avg_los
FROM inpatient_claims
WHERE CLM_PMT_AMT > 0
GROUP BY CLM_DRG_CD
ORDER BY total_cost DESC
LIMIT 20
""", conn)

print("Top 20 DRGs by Total Spending")
print("="*90)
display(drg_cost)

In [ ]:
drg_cost.to_csv("../data/exports/drg_cost_analysis.csv", index=False)
print("Saved: data/exports/drg_cost_analysis.csv")

---
## 2. Cost drivers by primary diagnosis (Inpatient)

In [ ]:
diag_cost = pd.read_sql("""
SELECT
  ICD9_DGNS_CD_1 as primary_diagnosis,
  COUNT(*) as num_claims,
  ROUND(SUM(CLM_PMT_AMT), 0) as total_cost,
  ROUND(AVG(CLM_PMT_AMT), 0) as avg_cost_per_claim
FROM inpatient_claims
WHERE CLM_PMT_AMT > 0 AND ICD9_DGNS_CD_1 IS NOT NULL
GROUP BY ICD9_DGNS_CD_1
ORDER BY total_cost DESC
LIMIT 20
""", conn)

print("Top 20 Primary Diagnoses by Total Spending (Inpatient)")
print("="*90)
display(diag_cost)
diag_cost.to_csv("../data/exports/diagnosis_cost_analysis.csv", index=False)

---
## 3. Impact of chronic conditions on annual spending

In [ ]:
chronic_impact = pd.DataFrame()

for condition in ['SP_DIABETES', 'SP_CHF', 'SP_COPD', 'SP_ISCHMCHT']:
    df = pd.read_sql(f"""
    SELECT
      CASE WHEN {condition} = 1 THEN COALESCE(MEDREIMB_IP, 0) + COALESCE(MEDREIMB_OP, 0) ELSE NULL END as spend_with,
      CASE WHEN {condition} = 2 THEN COALESCE(MEDREIMB_IP, 0) + COALESCE(MEDREIMB_OP, 0) ELSE NULL END as spend_without
    FROM beneficiary_summary
    """, conn)
    
    row = {
        'condition': condition,
        'avg_spend_with_condition': round(df['spend_with'].mean()),
        'avg_spend_without_condition': round(df['spend_without'].mean())
    }
    chronic_impact = pd.concat([chronic_impact, pd.DataFrame([row])], ignore_index=True)

print("Annual Spending Impact: With vs Without Chronic Conditions")
print("="*90)
display(chronic_impact)
chronic_impact.to_csv("../data/exports/chronic_condition_spend_impact.csv", index=False)

---
## 4. High-cost beneficiary profiles (top 10%)

In [ ]:
spending = pd.read_sql("""
SELECT
  DESYNPUF_ID,
  YEAR,
  COALESCE(MEDREIMB_IP, 0) + COALESCE(MEDREIMB_OP, 0) + COALESCE(MEDREIMB_CAR, 0) as total_medicare_spend,
  SP_DIABETES,
  SP_CHF,
  SP_COPD,
  SP_ISCHMCHT,
  SP_DEPRESSN,
  SP_ALZHDMTA
FROM beneficiary_summary
""", conn)

p90 = spending['total_medicare_spend'].quantile(0.90)
top_10pct = spending[spending['total_medicare_spend'] >= p90].copy()

print(f"90th percentile spending threshold: ${p90:,.0f}")
print(f"Top 10% of beneficiary-years: {len(top_10pct):,} (out of {len(spending):,})")
print(f"Top 10% avg spend: ${top_10pct['total_medicare_spend'].mean():,.0f}")
print(f"All beneficiaries avg spend: ${spending['total_medicare_spend'].mean():,.0f}")
print(f"\nChronic condition prevalence in top 10%:")
print(f"  Diabetes:           {(top_10pct['SP_DIABETES'] == 1).mean()*100:.1f}%")
print(f"  CHF:                {(top_10pct['SP_CHF'] == 1).mean()*100:.1f}%")
print(f"  COPD:               {(top_10pct['SP_COPD'] == 1).mean()*100:.1f}%")
print(f"  Ischemic Heart Dz:  {(top_10pct['SP_ISCHMCHT'] == 1).mean()*100:.1f}%")
print(f"  Depression:         {(top_10pct['SP_DEPRESSN'] == 1).mean()*100:.1f}%")

---
## 5. Spending trends by year

In [ ]:
yearly_trends = pd.read_sql("""
SELECT
  YEAR,
  COUNT(DISTINCT DESYNPUF_ID) as num_beneficiaries,
  ROUND(AVG(MEDREIMB_IP), 0) as avg_inpatient_spend,
  ROUND(AVG(MEDREIMB_OP), 0) as avg_outpatient_spend,
  ROUND(AVG(COALESCE(MEDREIMB_IP, 0) + COALESCE(MEDREIMB_OP, 0) + COALESCE(MEDREIMB_CAR, 0)), 0) as avg_total_spend
FROM beneficiary_summary
GROUP BY YEAR
ORDER BY YEAR
""", conn)

print("Spending Trends: 2008–2010")
print("="*90)
display(yearly_trends)
yearly_trends.to_csv("../data/exports/spending_trends_by_year.csv", index=False)

In [ ]:
conn.close()
print("
Analysis complete. Outputs saved to data/exports/")